In [1]:
import optuna
import numpy as np
import pandas as pd
import yfinance as yf
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

import gymnasium as gym
from gymnasium import spaces

import gc

import optuna
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from wandb.integration.sb3 import WandbCallback
import wandb
from alibi_detect.od import IForest

from stable_baselines3.common.callbacks import BaseCallback
import torch

d:\dev python\OrionTrader\.venv\lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
d:\dev python\OrionTrader\.venv\lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type ali

# Environnement Forex compatible Gymnasium

In [2]:
class ForexEnv(gym.Env):
    def __init__(self, df):
        super(ForexEnv, self).__init__()
        self.df = df.reset_index(drop=True)
        self.max_steps = len(df) - 1
        self.current_step = 0

        # Action : 0 = hold, 1 = buy, 2 = sell
        self.action_space = spaces.Discrete(3)

        # Observation : prix actuel et delta précédent
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(2,), dtype=np.float32)

        self.position = 0  # 0 = neutral, 1 = long, -1 = short
        self.entry_price = 0.0

    def reset(self, **kwargs):
        self.current_step = 0
        self.position = 0
        self.entry_price = 0.0
        obs = self._next_obs()
        info = {}
        return obs, info

    def _next_obs(self):
        price = float(self.df.loc[self.current_step, 'Close'])
        delta = float(self.df.loc[self.current_step, 'Close'] - self.df.loc[self.current_step-1, 'Close']) if self.current_step > 0 else 0.0
        obs = np.array([price, delta], dtype=np.float32)
        return obs

    # def step(self, action):
    #     prev_price = float(self.df.loc[self.current_step, 'Close'])
    #     delta = float(prev_price - self.df.loc[self.current_step-1, 'Close']) if self.current_step > 0 else 0.0

    #     # Mettre à jour la position
    #     if action == 1:  # buy
    #         self.position = 1
    #     elif action == 2:  # sell
    #         self.position = -1
    #     # hold (0) garde self.position inchangé

    #     # reward basé sur delta et position courante
    #     reward = delta * self.position * 1000  # multiplier pour une échelle plus grande

    #     self.current_step += 1
    #     terminated = self.current_step >= len(self.df) - 1
    #     truncated = False
    #     price = float(self.df.loc[self.current_step, 'Close']) if not terminated else prev_price
    #     obs = np.array([price, delta], dtype=np.float32)
    #     info = {}

    #     return obs, float(reward), terminated, truncated, info
    
    def step(self, action):
        prev_price = float(self.df.loc[self.current_step, 'Close'])
        self.current_step += 1
        terminated = self.current_step >= len(self.df) - 1
        price = float(self.df.loc[self.current_step, 'Close'])
        delta = price - prev_price

        # Actions : 0 hold, 1 buy, 2 sell, 3 close
        if action == 1:
            self.position = 1
        elif action == 2:
            self.position = -1
        elif action == 3:
            self.position = 0

        reward = delta * self.position * 1000  # amplifié pour stabilité

        obs = np.array([price, delta], dtype=np.float32)
        info = {}
        truncated = False
        return obs, reward, terminated, truncated, info




# Chargement des données EUR/USD

In [3]:
df = yf.download("EURUSD=X", start="2020-01-01", end="2023-01-01", interval="1d")
df = df.dropna().reset_index(drop=True)

C:\Users\Aurelien\AppData\Local\Temp\ipykernel_28464\1899308409.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("EURUSD=X", start="2020-01-01", end="2023-01-01", interval="1d")
[*********************100%***********************]  1 of 1 completed


# Fonction d'optimisation Optuna

In [4]:
def make_env(seed=None):
    def _init():
        env = ForexEnv(df)
        # Gymnasium reset peut accepter seed ; sinon on applique seeds globalement
        try:
            env.reset(seed=seed)
        except TypeError:
            pass
        return Monitor(env)
    return _init

# === OPTIMISATION DES HYPERPARAMÈTRES ===
def optimize_ppo(trial):
    import random, numpy as _np, torch as _torch
    seed = 1000 + trial.number
    random.seed(seed)
    _np.random.seed(seed)
    _torch.manual_seed(seed)

    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 5e-4)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    n_steps = max(batch_size, trial.suggest_categorical("n_steps", [1024, 2048, 4096]))
    gamma = trial.suggest_uniform("gamma", 0.90, 0.999)
    ent_coef = trial.suggest_uniform("ent_coef", 0.0, 0.02)

    # env d'entraînement (seedé) et env d'évaluation séparé
    train_env = DummyVecEnv([make_env(seed)])
    eval_env = DummyVecEnv([make_env(seed + 9999)])

    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        gamma=gamma,
        ent_coef=ent_coef,
        tensorboard_log="./logs/optuna_forex",
        verbose=0,
    )

    model.learn(total_timesteps=100_000, callback=NaNStopCallback())

    # evaluation sur env séparé, plus d'épisodes et non-deterministe pour plus de variance informative
    mean_reward, _ = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=False)

    tmp_path = f"tmp_models/trial_{trial.number}.zip"
    model.save(tmp_path)
    trial.set_user_attr("model_path", tmp_path)
    trial.set_user_attr("mean_reward", float(mean_reward))

    model.env.close()
    eval_env.close()
    del model, train_env, eval_env
    import gc; gc.collect()

    return float(mean_reward)

class NaNStopCallback(BaseCallback):
    def _on_step(self):
        for param in self.model.policy.parameters():
            if torch.isnan(param).any():
                return False  # stop training
        return True

# Lancer la recherche Optuna

In [5]:
study = optuna.create_study(direction="maximize")
study.optimize(optimize_ppo, n_trials=20)

best_trial = max(study.get_trials(deepcopy=False), key=lambda t: t.value if t.value is not None else float('-inf'))
best_model_path = best_trial.user_attrs.get("model_path")
from stable_baselines3 import PPO
if best_model_path:
    best_model = PPO.load(best_model_path)
else:
    # re-entraîner à partir des best_params
    best_params = best_trial.params
    env = DummyVecEnv([lambda: Monitor(ForexEnv(df))])
    best_model = PPO("MlpPolicy", env, **best_params)
    best_model.learn(total_timesteps=500_000)
print("✅ Best trial:")
print("  Reward:", best_trial.value)
print("  Params:", best_trial.params)

[I 2025-10-24 16:32:53,495] A new study created in memory with name: no-name-b438ffab-767e-4a5b-8c99-e09134bc166e
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_28464\3768260216.py:20: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 5e-4)
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_28464\3768260216.py:23: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  gamma = trial.suggest_uniform("gamma", 0.90, 0.999)
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_28464\3768260216.py:24: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.

✅ Best trial:
  Reward: 132.40696210000002
  Params: {'learning_rate': 5.7349821681901945e-05, 'batch_size': 256, 'n_steps': 1024, 'gamma': 0.9329876690929497, 'ent_coef': 0.005132255597279512}


In [6]:
# === ENTRAÎNEMENT FINAL AVEC W&B ===
best_params = best_trial.params
# Correction ici : on passe la fonction make_env() et non make_env
env = DummyVecEnv([make_env()])  

wandb.init(
    mode="offline", 
    project="forex-ppo-hallucination",
    config=best_params,
)

best_model = PPO("MlpPolicy", env, **best_params, verbose=0)
best_model.learn(
    total_timesteps=500_000,
    callback=WandbCallback(
        gradient_save_freq=100,
        model_save_path=f"models/{wandb.run.name}",
        verbose=2,
    ),
)
best_model.save("models/best_ppo_forex.zip")
wandb.finish()

C:\Users\Aurelien\AppData\Local\Temp\ipykernel_28464\280050115.py:26: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(self.df.loc[self.current_step, 'Close'])


C:\Users\Aurelien\AppData\Local\Temp\ipykernel_28464\280050115.py:55: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  prev_price = float(self.df.loc[self.current_step, 'Close'])
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_28464\280050115.py:58: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(self.df.loc[self.current_step, 'Close'])


# FONCTION DE DÉTECTION D’HALLUCINATIONS

In [7]:
def detect_hallucinations(model, env, n_steps=5000, threshold=0.05):
    obs = env.reset()
    actions = []
    rewards = []

    for _ in range(n_steps):
        action, _ = model.predict(obs, deterministic=False)
        obs, reward, done, info = env.step(action)
        actions.append(action.flatten())
        rewards.append(np.mean(reward))
        if done:
            obs = env.reset()

    actions = np.array(actions)
    rewards = np.array(rewards)

    # === Détection d’anomalies sur les actions ===
    od = IForest(threshold=threshold)
    od.fit(actions)
    preds = od.predict(actions)

    anomaly_ratio = preds["data"]["is_outlier"].mean()
    print(f"⚠️ Anomaly ratio (hallucination risk): {anomaly_ratio:.2%}")

    if anomaly_ratio > 0.1:
        print("🚨 Possible policy hallucination detected! (actions incohérentes)")
    else:
        print("✅ Policy seems stable and consistent.")

    return anomaly_ratio

# ÉVALUATION DU COMPORTEMENT DU MODÈLE

In [8]:
env_test = DummyVecEnv([lambda: Monitor(ForexEnv(df))])

detect_hallucinations(best_model, env_test, n_steps=5000, threshold=0.05)
env.close()


C:\Users\Aurelien\AppData\Local\Temp\ipykernel_28464\280050115.py:26: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(self.df.loc[self.current_step, 'Close'])
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_28464\280050115.py:55: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  prev_price = float(self.df.loc[self.current_step, 'Close'])
C:\Users\Aurelien\AppData\Local\Temp\ipykernel_28464\280050115.py:58: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  price = float(self.df.loc[self.current_step, 'Close'])


⚠️ Anomaly ratio (hallucination risk): 4.66%
✅ Policy seems stable and consistent.
